In [1]:
import numpy as np
import pandas as pd
import torch
%load_ext autoreload
%autoreload 2

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print (device)

cuda:0


In [2]:
train_path = './data/new_train.tsv'
test_path = './data/new_test.tsv'
df_train = pd.read_csv(train_path, sep = '\t', header=None)
df_test = pd.read_csv(test_path, sep = '\t', header=None)
x_train = list(df_train[0])
y_train = torch.tensor((df_train[1])).to(device)
x_test = list(df_test[0])
y_test = torch.tensor(df_test[1]).to(device)

# print ("x_train", x_train[0:2])
# print ("y_train", y_train[0:2])
# df_train.head()

In [3]:
"""
1. Bag of Words：建立词袋
"""
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(x_train)
X_test = vectorizer.transform(x_test)
X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test.toarray(), dtype=torch.float32).to(device)
# print (X_train[:1])
# print (len(X_train_tensor[0]))
# print (len(vectorizer.vocabulary_))

In [4]:
# print("y_train shape:", y_train.shape)
# print("y_train dtype:", y_train.dtype)
# print("y_train sample:", y_train[:5])

In [5]:
"""
BoW的训练与可视化
"""
from utils import Model
from utils import crossentropy
from utils import evaluate
import os
import shutil
from torch.utils.tensorboard import SummaryWriter

# 用 tensorboard 记录结果
log_dir = "runs/BoW"
if os.path.exists(log_dir):
    shutil.rmtree(log_dir)  # 删除旧日志，避免曲线混淆
writer = SummaryWriter(log_dir)

INPUT_SIZE = len(X_train_tensor[0])
HIDDEN_SIZE = 128
OUTPUT_SIZE = 5
BATCH_SIZE = 128
EPOCHS = 150
LR = 0.005
SHUFFLE = True
num_samples = X_train_tensor.shape[0]

model = Model(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, lr=LR)

for epoch in range(EPOCHS):
    # shuffle
    indices = torch.randperm(num_samples)
    X_shuffled = X_train_tensor[indices]
    y_shuffled = y_train[indices]

    epoch_loss_sum = 0.0

    for i in range(0, num_samples, BATCH_SIZE):
        X_batch = X_shuffled[i : i + BATCH_SIZE]
        y_batch = y_shuffled[i : i + BATCH_SIZE]
        
        y_pre = model.forward(X_batch)
        loss = crossentropy(y_pre, y_batch)
        model.backward(y_pre, y_batch)

        epoch_loss_sum += loss.item()


    if (epoch + 1) % 10 == 0:
        train_loss, train_acc = evaluate(model, X_train_tensor, y_train)
        test_loss, test_acc = evaluate(model, X_test_tensor, y_test)
        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f} | "
              f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}")

        # 写入 tensorboard
        writer.add_scalar('Loss/Train', train_loss, epoch+1)
        writer.add_scalar('Loss/Test', test_loss, epoch+1)
        writer.add_scalar('Accuracy/Train', train_acc, epoch+1)
        writer.add_scalar('Accuracy/Test', test_acc, epoch+1)

writer.close()
print ("Traing Over")

Epoch [10/150] | Train Loss: 1.4937, Acc: 0.36 | Test Loss: 1.4987, Acc: 0.35
Epoch [20/150] | Train Loss: 1.4611, Acc: 0.38 | Test Loss: 1.4705, Acc: 0.37
Epoch [30/150] | Train Loss: 1.4175, Acc: 0.41 | Test Loss: 1.4360, Acc: 0.39
Epoch [40/150] | Train Loss: 1.3569, Acc: 0.46 | Test Loss: 1.3937, Acc: 0.42
Epoch [50/150] | Train Loss: 1.2767, Acc: 0.50 | Test Loss: 1.3450, Acc: 0.45
Epoch [60/150] | Train Loss: 1.1814, Acc: 0.54 | Test Loss: 1.2983, Acc: 0.46
Epoch [70/150] | Train Loss: 1.0785, Acc: 0.58 | Test Loss: 1.2611, Acc: 0.48
Epoch [80/150] | Train Loss: 0.9736, Acc: 0.62 | Test Loss: 1.2353, Acc: 0.48
Epoch [90/150] | Train Loss: 0.8586, Acc: 0.68 | Test Loss: 1.2200, Acc: 0.49
Epoch [100/150] | Train Loss: 0.7556, Acc: 0.73 | Test Loss: 1.2286, Acc: 0.50
Epoch [110/150] | Train Loss: 0.6488, Acc: 0.79 | Test Loss: 1.2637, Acc: 0.49
Epoch [120/150] | Train Loss: 0.5454, Acc: 0.84 | Test Loss: 1.2911, Acc: 0.50
Epoch [130/150] | Train Loss: 0.4534, Acc: 0.87 | Test Loss: 

In [6]:
"""
2. N-gram 建立词袋
"""
N = 2

ngram_vectorizer = CountVectorizer(ngram_range=(1, N))

X_train_ngram = ngram_vectorizer.fit_transform(x_train)
X_test_ngram = ngram_vectorizer.transform(x_test)
X_train_tensor = torch.tensor(X_train_ngram.toarray(), dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_ngram.toarray(), dtype=torch.float32).to(device)

# print (len(X_train_tensor))
# print("N-gram 词典:", ngram_vectorizer.get_feature_names_out()[:5])
# print ("len N-gram", len(ngram_vectorizer.get_feature_names_out()))

In [7]:
"""
N-gram 的训练与可视化
"""
from utils import Model
from utils import crossentropy
from utils import evaluate
import os
import shutil
from torch.utils.tensorboard import SummaryWriter

# 用 tensorboard 记录结果
log_dir = "runs/N-gram-n_2"
if os.path.exists(log_dir):
    shutil.rmtree(log_dir)  # 删除旧日志，避免曲线混淆
writer = SummaryWriter(log_dir)

INPUT_SIZE = len(X_train_tensor[0])
HIDDEN_SIZE = 128
OUTPUT_SIZE = 5
BATCH_SIZE = 128
EPOCHS = 100
LR = 0.005
SHUFFLE = True
num_samples = X_train_tensor.shape[0]

model_ngram = Model(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, lr=LR)

for epoch in range(EPOCHS):
    # shuffle
    indices = torch.randperm(num_samples)
    X_shuffled = X_train_tensor[indices]
    y_shuffled = y_train[indices]

    epoch_loss_sum = 0.0

    for i in range(0, num_samples, BATCH_SIZE):
        X_batch = X_shuffled[i : i + BATCH_SIZE]
        y_batch = y_shuffled[i : i + BATCH_SIZE]
        
        y_pre = model_ngram.forward(X_batch)
        loss = crossentropy(y_pre, y_batch)
        model_ngram.backward(y_pre, y_batch)

        epoch_loss_sum += loss.item()


    if (epoch + 1) % 10 == 0:
        train_loss, train_acc = evaluate(model_ngram, X_train_tensor, y_train)
        test_loss, test_acc = evaluate(model_ngram, X_test_tensor, y_test)
        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f} | "
              f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}")

        # 写入 tensorboard
        writer.add_scalar('Loss/Train', train_loss, epoch+1)
        writer.add_scalar('Loss/Test', test_loss, epoch+1)
        writer.add_scalar('Accuracy/Train', train_acc, epoch+1)
        writer.add_scalar('Accuracy/Test', test_acc, epoch+1)

writer.close()
print ("Traing Over")

Epoch [10/100] | Train Loss: 1.4603, Acc: 0.39 | Test Loss: 1.4766, Acc: 0.38
Epoch [20/100] | Train Loss: 1.3790, Acc: 0.46 | Test Loss: 1.4262, Acc: 0.41
Epoch [30/100] | Train Loss: 1.2441, Acc: 0.54 | Test Loss: 1.3613, Acc: 0.45
Epoch [40/100] | Train Loss: 1.0408, Acc: 0.63 | Test Loss: 1.2934, Acc: 0.47
Epoch [50/100] | Train Loss: 0.7960, Acc: 0.75 | Test Loss: 1.2451, Acc: 0.49
Epoch [60/100] | Train Loss: 0.5657, Acc: 0.85 | Test Loss: 1.2359, Acc: 0.49
Epoch [70/100] | Train Loss: 0.3870, Acc: 0.90 | Test Loss: 1.2651, Acc: 0.49
Epoch [80/100] | Train Loss: 0.2639, Acc: 0.94 | Test Loss: 1.3111, Acc: 0.49
Epoch [90/100] | Train Loss: 0.1818, Acc: 0.97 | Test Loss: 1.3699, Acc: 0.49
Epoch [100/100] | Train Loss: 0.1287, Acc: 0.99 | Test Loss: 1.4365, Acc: 0.49
Traing Over


In [9]:
N = 3

ngram_vectorizer = CountVectorizer(ngram_range=(1, N))

X_train_ngram = ngram_vectorizer.fit_transform(x_train)
X_test_ngram = ngram_vectorizer.transform(x_test)
X_train_tensor = torch.tensor(X_train_ngram.toarray(), dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_ngram.toarray(), dtype=torch.float32).to(device)


# 用 tensorboard 记录结果
log_dir = "runs/N-gram-n_3"
if os.path.exists(log_dir):
    shutil.rmtree(log_dir)  # 删除旧日志，避免曲线混淆
writer = SummaryWriter(log_dir)

INPUT_SIZE = len(X_train_tensor[0])
HIDDEN_SIZE = 128
OUTPUT_SIZE = 5
BATCH_SIZE = 128
EPOCHS = 100
LR = 0.005
SHUFFLE = True
num_samples = X_train_tensor.shape[0]

model_ngram = Model(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, lr=LR)

for epoch in range(EPOCHS):
    # shuffle
    indices = torch.randperm(num_samples)
    X_shuffled = X_train_tensor[indices]
    y_shuffled = y_train[indices]

    epoch_loss_sum = 0.0

    for i in range(0, num_samples, BATCH_SIZE):
        X_batch = X_shuffled[i : i + BATCH_SIZE]
        y_batch = y_shuffled[i : i + BATCH_SIZE]
        
        y_pre = model_ngram.forward(X_batch)
        loss = crossentropy(y_pre, y_batch)
        model_ngram.backward(y_pre, y_batch)

        epoch_loss_sum += loss.item()


    if (epoch + 1) % 10 == 0:
        train_loss, train_acc = evaluate(model_ngram, X_train_tensor, y_train)
        test_loss, test_acc = evaluate(model_ngram, X_test_tensor, y_test)
        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f} | "
              f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.2f}")

        # 写入 tensorboard
        writer.add_scalar('Loss/Train', train_loss, epoch+1)
        writer.add_scalar('Loss/Test', test_loss, epoch+1)
        writer.add_scalar('Accuracy/Train', train_acc, epoch+1)
        writer.add_scalar('Accuracy/Test', test_acc, epoch+1)

writer.close()
print ("Traing Over")

OutOfMemoryError: CUDA out of memory. Tried to allocate 6.49 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 12.33 GiB is allocated by PyTorch, and 896.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)